In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
import pandas as pd
import time
import os


# -------------------------------------------------------------
# 初始化浏览器
# -------------------------------------------------------------
def init_browser():
    print("初始化浏览器...")

    options = webdriver.ChromeOptions()
    options.add_argument('--disable-blink-features=AutomationControlled')
    options.add_argument('--start-maximized')
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option('useAutomationExtension', False)

    driver = webdriver.Chrome(options=options)
    driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")

    return driver



#To find the list,tried 5 kinds of selector.Class_name do work.

In [2]:
def find_list_items(driver, saved_selector):
    print("查找地震列表项...")

    # 若已有可复用的选择器，直接使用
    if saved_selector["selector"] and saved_selector["type"]:
        try:
            items = driver.find_elements(saved_selector["type"], saved_selector["selector"])
            if items:
                valid = [i for i in items if i.text.strip()]
                if valid:
                    print(f"使用已保存选择器找到 {len(valid)} 个列表项")
                    return valid
        except:
            print("已保存选择器失效，重新查找...")

    # 多种可能的选择器
    list_item_selectors = [
        (By.CLASS_NAME, "DesignatedCatalogue_list-item__axM47"),
        # (By.CSS_SELECTOR, "[class*='list-item']"),
        # (By.XPATH, "//div[contains(@class, 'list-item')]"),
        # (By.XPATH, "//div[contains(text(), 'M') or contains(text(), '级')]"),
        # (By.XPATH, "//div[contains(@class, 'DesignatedCatalogue')]"),
    ]

    for selector_type, selector_value in list_item_selectors:
        try:
            items = driver.find_elements(selector_type, selector_value)
            valid_items = [i for i in items if i.text.strip()]
            if valid_items:
                print(f"选择器 '{selector_value}' 找到 {len(valid_items)} 项")
                saved_selector["selector"] = selector_value
                saved_selector["type"] = selector_type
                return valid_items
        except:
            continue

    print("未找到任何地震列表项")
    return []

#To extract features,after click the current list.Finding the time,Longitude and Latitude，Hypocentral Depth，Earthquake Magnitude
#the second time add the feature:title

In [3]:
def extract_features(driver, count, item_text):
    xpath = '//*[@id="root"]/div/div/div/div[2]/div/div[2]/div/div/div/div[1]/div[2]'

    try:
        container = driver.find_element(By.XPATH, xpath)
        lines = [l.strip() for l in container.text.split("\n") if l.strip()]

        if len(lines) >= 4:
            return {
                '序号': count + 1,
                '列表项文本': item_text,    # ←★ 保存新增字段
                '时间': lines[0],
                '经纬度': lines[1],
                '震源深度': lines[2],
                '级数': lines[3]
            }
        else:
            print("    ✗ 内容不足 4 行")
            return None

    except Exception as e:
        print(f"    特征提取失败: {e}")
        return None


#Whether it is affected by network speed or not, the sleep time must be more than 5 seconds; otherwise, the content will fail to load, leading to the failure to locate the element.

In [4]:
def wait_for_list_page(driver, saved_selector):
    print("等待列表页面加载...")
    time.sleep(7)

    for _ in range(5):
        items = find_list_items(driver, saved_selector)
        if items:
            print("列表已加载")
            return True
        time.sleep(1)

    print("列表页面加载失败")
    return False

#put 'item_text' into features

In [5]:
def click_and_extract(driver, item, index, data_list):
    print(f"\n正在处理第 {index+1} 个地震...")

    try:
        # 新增：列表项文本
        item_text = item.text.strip()
        print(f"  列表项文本: {item_text}")

        print("  点击列表项...")
        item.click()
        time.sleep(3)

        print("  提取详情页特征...")

        # ★ 把 item_text 传入 extract_features
        features = extract_features(driver, len(data_list), item_text)

        if features:
            data_list.append(features)
            print(f"  ✓ 成功提取 {features['时间']} | {features['级数']} 级")
            return True
        else:
            print("  ✗ 提取失败")
            return False

    except Exception as e:
        print(f"  处理失败: {e}")
        return False


#Do not reload the list before crawling reaches the last max_items. Reloading takes a long time and is prone to errors. Pause for 5 seconds to prevent anti-crawling measures.
#The first crawl was conducted on December 9, with a total count of 1,344. The second crawl took place on December 12, and the count increased to 1,356.


In [6]:
def crawl_earthquakes(driver, max_items=1356):
    saved_selector = {"selector": None, "type": None}
    data_list = []

    list_url = "https://www.cenc.ac.cn/earthquake-manage-publish-web/designated-catalogue"
    print(f"\n访问列表页: {list_url}")
    driver.get(list_url)

    if not wait_for_list_page(driver, saved_selector):
        return data_list

    items = find_list_items(driver, saved_selector)
    items_to_crawl = min(len(items), max_items)

    for i in range(items_to_crawl):
        items_now = find_list_items(driver, saved_selector)
        if i >= len(items_now):
            break

        click_and_extract(driver, items_now[i], i, data_list)

        # if i < items_to_crawl - 1:
        #     driver.back()
        #     wait_for_list_page(driver, saved_selector)

        time.sleep(5)

    print(f"\n共获取 {len(data_list)} 条记录")
    return data_list

In [7]:
def save_to_excel(data):
    if not data:
        print("没有数据可保存")
        return

    desktop = os.path.join(os.path.expanduser('~'), 'Desktop')
    filename = os.path.join(
        desktop,
        f"地震详情数据_{time.strftime('%Y%m%d_%H%M%S')}.xlsx"
    )

    df = pd.DataFrame(data)

    # 增加新列排序
    column_order = ['序号', '列表项文本', '时间', '经纬度', '震源深度', '级数']
    df = df[[col for col in column_order if col in df.columns]]

    df.to_excel(filename, index=False)
    print(f"已保存：{filename}")



In [8]:
def main():
    print("="*60)
    print("中国地震网数据采集")
    print("="*60)

    driver = init_browser()

    try:
        max_items = input("爬取数量（默认5）: ").strip()
        max_items = int(max_items) if max_items.isdigit() else 5

        data = crawl_earthquakes(driver, max_items)

        if data:
            save = input("是否保存 Excel? (y/n 默认y): ").strip().lower()
            if save != "n":
                save_to_excel(data)

    finally:
        driver.quit()
        print("浏览器已关闭")

In [9]:
if __name__ == "__main__":
    main()

中国地震网数据采集
初始化浏览器...


Error sending stats to Plausible: error sending request for url (https://plausible.io/api/event)



访问列表页: https://www.cenc.ac.cn/earthquake-manage-publish-web/designated-catalogue
等待列表页面加载...
查找地震列表项...
选择器 'DesignatedCatalogue_list-item__axM47' 找到 1356 项
列表已加载
查找地震列表项...
使用已保存选择器找到 1356 个列表项
查找地震列表项...
使用已保存选择器找到 1356 个列表项

正在处理第 1 个地震...
  列表项文本: 2025-12-12 22:22:11新疆和田地区皮山县3.9级地震
  点击列表项...
  提取详情页特征...
  ✓ 成功提取 2025-12-12 22:22:11 | 3.9级 级
查找地震列表项...
使用已保存选择器找到 1356 个列表项

正在处理第 2 个地震...
  列表项文本: 2025-12-12 11:04:55新疆吐鲁番市托克逊县3.2级地震
  点击列表项...
  提取详情页特征...
  ✓ 成功提取 2025-12-12 11:04:55 | 3.2级 级
查找地震列表项...
使用已保存选择器找到 1356 个列表项

正在处理第 3 个地震...
  列表项文本: 2025-12-12 10:44:11日本本州东部附近海域6.8级地震
  点击列表项...
  提取详情页特征...
  ✓ 成功提取 2025-12-12 10:44:11 | 6.8级 级
查找地震列表项...
使用已保存选择器找到 1356 个列表项

正在处理第 4 个地震...
  列表项文本: 2025-12-12 01:18:31云南昭通市大关县3.9级地震
  点击列表项...
  提取详情页特征...
  ✓ 成功提取 2025-12-12 01:18:31 | 3.9级 级
查找地震列表项...
使用已保存选择器找到 1356 个列表项

正在处理第 5 个地震...
  列表项文本: 2025-12-11 21:14:26西藏那曲市双湖县3.1级地震
  点击列表项...
  提取详情页特征...
  ✓ 成功提取 2025-12-11 21:14:26 | 3.1级 级
查找地震列表项...
使用已保存选择器找到 1356 个列表项

正